# 02. Ingestão Dinâmica de Consistência Mensal e Anual para o Supabase

Este notebook localiza dinamicamente as planilhas mais recentes nas pastas de saída oficiais do Elabore:
- `data/outputs/monthly/` (*Indicadores Mensais*)
- `data/outputs/annual/` (*Indicadores Anuais*)

Em seguida, faz o de-para de `codigo_lr` $
ightarrow$ `idfazenda` (inteiro) consultando a `tab_fazenda` do Supabase, formata as datas e executa a carga via **UPSERT** no Supabase nas tabelas:
- `tab_consistencia_mensal`
- `tab_consistencia_anual`

In [1]:
# 1. Setup do Ambiente e Conexão com o Supabase
from __future__ import annotations

import os
import sys
import glob
import warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

warnings.filterwarnings('ignore', category=FutureWarning)

# Carregar variáveis de ambiente (.env do projeto elabore-views)
raiz_projeto = Path.cwd().resolve()
for candidato in [raiz_projeto, *raiz_projeto.parents]:
    if (candidato / ".env").exists():
        load_dotenv(candidato / ".env")
        raiz_projeto = candidato
        break
else:
    load_dotenv()

supabase_url = os.getenv('SUPABASE_URL')
supabase_key = os.getenv('SUPABASE_SERVICE_KEY')
if not supabase_url or not supabase_key:
    raise ValueError("Variáveis SUPABASE_URL e SUPABASE_SERVICE_KEY devem estar configuradas no .env.")

supabase = create_client(supabase_url, supabase_key)
print(f"✅ Conexão com o Supabase inicializada! Raiz do projeto: {raiz_projeto}")

# Função para carregar o mapa de de-para codigo_lr -> idfazenda (int) a partir da tab_fazenda
def obter_mapa_codigo_lr_idfazenda():
    try:
        res = supabase.table('tab_fazenda').select('id, codAgroindustria').execute()
        if res.data:
            df_fz = pd.DataFrame(res.data)
            df_fz.dropna(subset=['codAgroindustria', 'id'], inplace=True)
            # Tratar id para int e remover espaços em codAgroindustria
            df_fz['codAgroindustria'] = df_fz['codAgroindustria'].astype(str).str.strip()
            df_fz['id'] = pd.to_numeric(df_fz['id'], errors='coerce')
            df_fz.dropna(subset=['id'], inplace=True)
            return dict(zip(df_fz['codAgroindustria'], df_fz['id'].astype(int)))
    except Exception as e:
        print(f"❌ Erro ao buscar mapa tab_fazenda: {e}")
    return {}

mapa_lr_fazenda = obter_mapa_codigo_lr_idfazenda()
print(f"✅ Mapa de de-para codigo_lr -> idfazenda carregado: {len(mapa_lr_fazenda)} fazendas mapeadas.")

✅ Conexão com o Supabase inicializada! Raiz do projeto: C:\Users\Guilherme\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\elabore-views
✅ Mapa de de-para codigo_lr -> idfazenda carregado: 1094 fazendas mapeadas.


## 1. Processamento e Ingestão: `tab_consistencia_mensal`

In [2]:
# Busca dinâmica do arquivo de indicadores mensais mais recente
pastas_mensais = [
    raiz_projeto / "data" / "outputs" / "monthly",
    Path(r"C:\Users\Guilherme\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\lr-smartquestion\DB\INPUT\TEMP"),
    raiz_projeto / "data" / "OUTPUT" / "PROCESSED",
    raiz_projeto / "app",
    raiz_projeto
]

candidatos_mensais = []
for p in pastas_mensais:
    if p.exists():
        for f in p.glob("*.xlsx"):
            n = f.name.lower()
            if ('mensa' in n or 'monthly' in n) and not ('anua' in n or 'annual' in n):
                candidatos_mensais.append(f)

candidatos_mensais = list(set(candidatos_mensais))
if not candidatos_mensais:
    raise FileNotFoundError("Nenhum arquivo .xlsx de indicadores mensais foi localizado.")

arquivo_mensal = max(candidatos_mensais, key=os.path.getmtime)
print(f"📄 Arquivo mensal dinâmico selecionado: {arquivo_mensal}")

# Leitura dos dados
df_m = pd.read_excel(arquivo_mensal)

# Mapeamento completo (Português / Inglês / Variantes)
map_m = {
    'Código LR': 'codigo_lr', 'labor_rural_code': 'codigo_lr', 'codAgroindustria': 'codigo_lr',
    'Status de Consistência': 'consistencia_mensal', 'monthly_consistency_status': 'consistencia_mensal', 'Consistencia': 'consistencia_mensal',
    'Status Cadastral': 'status_code', 'monthly_status_annual': 'status_code', 'status_dados': 'status_code',
    'Mês de Referência': 'mes_elabore', 'reference_month': 'mes_elabore', 'mesReferencia': 'mes_elabore'
}

renames_m = {col: map_m[col] for col in df_m.columns if col in map_m}
df_cons_m = df_m.rename(columns=renames_m).copy()

# Mapear idfazenda (inteiro) a partir de codigo_lr usando mapa_lr_fazenda da tab_fazenda
if 'codigo_lr' in df_cons_m.columns:
    df_cons_m['codigo_lr_clean'] = df_cons_m['codigo_lr'].astype(str).str.strip()
    df_cons_m['idfazenda'] = df_cons_m['codigo_lr_clean'].map(mapa_lr_fazenda)
    df_cons_m['idfazenda'] = pd.to_numeric(df_cons_m['idfazenda'], errors='coerce').astype('Int64')

# Formatação e tratamento de datas
df_cons_m['mes_elabore_dt'] = pd.to_datetime(df_cons_m['mes_elabore'], errors='coerce')
df_cons_m['mes_referencia_dt'] = df_cons_m['mes_elabore_dt'] + pd.DateOffset(months=1)
df_cons_m['mes_elabore'] = df_cons_m['mes_elabore_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')
df_cons_m['mes_referencia'] = df_cons_m['mes_referencia_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')

# Seleção final de colunas
cols_m = ['idfazenda', 'codigo_lr', 'status_code', 'consistencia_mensal', 'mes_elabore', 'mes_referencia']
df_cons_m = df_cons_m[[c for c in cols_m if c in df_cons_m.columns]].copy()

# Descartar apenas registros sem idfazenda mapeado
nulos_finais = df_cons_m['idfazenda'].isna()
if nulos_finais.sum() > 0:
    print(f"⚠️ Descartando {nulos_finais.sum()} registros que não possuem correspondência de idfazenda na tab_fazenda.")
    df_cons_m = df_cons_m[~nulos_finais].copy()

df_cons_m.drop_duplicates(subset=['idfazenda', 'mes_referencia'], keep='last', inplace=True)

# Substituir NaN por None para JSON
for col in df_cons_m.columns:
    df_cons_m.loc[df_cons_m[col].isna(), col] = None

records_m = df_cons_m.to_dict(orient='records')
for r in records_m:
    if 'idfazenda' in r and pd.notna(r['idfazenda']):
        r['idfazenda'] = int(r['idfazenda'])

print(f"📊 Total de registros mensais para UPSERT: {len(records_m)}")

# Carga via UPSERT em lotes de 1.000
chunk_size = 1000
total_upserted_m = 0
for i in range(0, len(records_m), chunk_size):
    chunk = records_m[i:i + chunk_size]
    try:
        res = supabase.table('tab_consistencia_mensal').upsert(chunk).execute()
        if res.data:
            total_upserted_m += len(res.data)
            print(f"   - Lote {i//chunk_size + 1}: {len(res.data)} registros inseridos/atualizados.")
    except Exception as e:
        print(f"❌ Erro no lote {i//chunk_size + 1}: {e}")

print(f"✅ Processamento da consistência mensal concluído. Total upserted: {total_upserted_m}")

📄 Arquivo mensal dinâmico selecionado: C:\Users\Guilherme\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\elabore-views\data\outputs\monthly\2026_08_06_085617_indicadores_mensais.xlsx
⚠️ Descartando 409 registros que não possuem correspondência de idfazenda na tab_fazenda.
📊 Total de registros mensais para UPSERT: 12895
   - Lote 1: 1000 registros inseridos/atualizados.
   - Lote 2: 1000 registros inseridos/atualizados.
   - Lote 3: 1000 registros inseridos/atualizados.
   - Lote 4: 1000 registros inseridos/atualizados.
   - Lote 5: 1000 registros inseridos/atualizados.
   - Lote 6: 1000 registros inseridos/atualizados.
   - Lote 7: 1000 registros inseridos/atualizados.
   - Lote 8: 1000 registros inseridos/atualizados.
   - Lote 9: 1000 registros inseridos/atualizados.
   - Lote 10: 1000 registros inseridos/atualizados.
   - Lote 11: 1000 registros inseridos/atualizados.
   - Lote 12: 1000 registros inseridos/atualizados.
   - Lote 13: 895 registros inse

In [9]:
nulos_finais[nulos_finais == True]

17       True
18       True
19       True
20       True
21       True
         ... 
13029    True
13030    True
13031    True
13032    True
13215    True
Name: idfazenda, Length: 409, dtype: bool

## 2. Processamento e Ingestão: `tab_consistencia_anual`

In [12]:
# Busca dinâmica do arquivo de indicadores anuais mais recente
pastas_anuais = [
    raiz_projeto / "data" / "outputs" / "annual",
    Path(r"C:\Users\Guilherme\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\lr-smartquestion\DB\INPUT\TEMP"),
    raiz_projeto / "data" / "OUTPUT" / "PROCESSED",
    raiz_projeto / "app",
    raiz_projeto
]

candidatos_anuais = []
for p in pastas_anuais:
    if p.exists():
        for f in p.glob("*.xlsx"):
            n = f.name.lower()
            if ('anua' in n or 'annual' in n) and not ('mensa' in n or 'monthly' in n):
                candidatos_anuais.append(f)

candidatos_anuais = list(set(candidatos_anuais))
if not candidatos_anuais:
    raise FileNotFoundError("Nenhum arquivo .xlsx de indicadores anuais foi localizado.")

# Selecionar dinamicamente o arquivo mais recente pelo timestamp de modificação
arquivo_anual = max(candidatos_anuais, key=os.path.getmtime)
print(f"📄 Arquivo anual dinâmico selecionado: {arquivo_anual}")

# Leitura dos dados
df_a = pd.read_excel(arquivo_anual)

# Mapeamento completo (Português / Inglês / Variantes)
map_a = {
    'Código LR': 'codigo_lr', 'labor_rural_code': 'codigo_lr', 'codAgroindustria': 'codigo_lr',
    'Status de Consistência': 'consistencia_anual', 'annual_consistency_status': 'consistencia_anual', 'Consistencia': 'consistencia_anual',
    'Mês de Referência': 'mes_elabore', 'annual_period_start': 'mes_elabore',
    'Fim de Referência': 'mes_referencia', 'annual_period_end': 'mes_referencia'
}

renames_a = {col: map_a[col] for col in df_a.columns if col in map_a}
df_cons_a = df_a.rename(columns=renames_a).copy()

# Mapear idfazenda (inteiro) a partir de codigo_lr usando mapa_lr_fazenda
if 'codigo_lr' in df_cons_a.columns:
    df_cons_a['codigo_lr_clean'] = df_cons_a['codigo_lr'].astype(str).str.strip()
    df_cons_a['idfazenda'] = df_cons_a['codigo_lr_clean'].map(mapa_lr_fazenda)
    df_cons_a['idfazenda'] = pd.to_numeric(df_cons_a['idfazenda'], errors='coerce').astype('Int64')

# Tratamento de datas
if 'mes_elabore' in df_cons_a.columns:
    df_cons_a['mes_elabore_dt'] = pd.to_datetime(df_cons_a['mes_elabore'], errors='coerce')
    df_cons_a['mes_elabore'] = df_cons_a['mes_elabore_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')

if 'mes_referencia' in df_cons_a.columns:
    df_cons_a['mes_referencia_dt'] = pd.to_datetime(df_cons_a['mes_referencia'], errors='coerce')
    df_cons_a['mes_referencia'] = df_cons_a['mes_referencia_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')
elif 'mes_elabore_dt' in df_cons_a.columns:
    df_cons_a['mes_referencia_dt'] = df_cons_a['mes_elabore_dt'] + pd.DateOffset(months=1)
    df_cons_a['mes_referencia'] = df_cons_a['mes_referencia_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')

# Seleção final de colunas
cols_a = ['idfazenda', 'codigo_lr', 'consistencia_anual', 'mes_elabore', 'mes_referencia']
df_cons_a = df_cons_a[[c for c in cols_a if c in df_cons_a.columns]].copy()

# Descartar apenas registros sem idfazenda mapeado
nulos_finais_a = df_cons_a['idfazenda'].isna()
if nulos_finais_a.sum() > 0:
    print(f"⚠️ Descartando {nulos_finais_a.sum()} registros anuais sem idfazenda válido.")
    df_cons_a = df_cons_a[~nulos_finais_a].copy()

df_cons_a.drop_duplicates(subset=['idfazenda', 'mes_elabore'], keep='last', inplace=True)

# Substituir NaN por None
for col in df_cons_a.columns:
    df_cons_a.loc[df_cons_a[col].isna(), col] = None

records_a = df_cons_a.to_dict(orient='records')
for r in records_a:
    if 'idfazenda' in r and pd.notna(r['idfazenda']):
        r['idfazenda'] = int(r['idfazenda'])

print(f"📊 Total de registros anuais para UPSERT: {len(records_a)}")

# Carga via UPSERT em lotes de 1.000
total_upserted_a = 0
for i in range(0, len(records_a), chunk_size):
    chunk = records_a[i:i + chunk_size]
    try:
        res = supabase.table('tab_consistencia_anual').upsert(chunk).execute()
        if res.data:
            total_upserted_a += len(res.data)
            print(f"   - Lote {i//chunk_size + 1}: {len(res.data)} registros inseridos/atualizados.")
    except Exception as e:
        print(f"❌ Erro no lote {i//chunk_size + 1}: {e}")

print(f"✅ Processamento da consistência anual concluído. Total upserted: {total_upserted_a}")

📄 Arquivo anual dinâmico selecionado: C:\Users\Guilherme\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\elabore-views\data\outputs\annual\2026_08_06_085726_indicadores_anuais.xlsx
⚠️ Descartando 73 registros anuais sem idfazenda válido.
📊 Total de registros anuais para UPSERT: 5392
   - Lote 1: 1000 registros inseridos/atualizados.
   - Lote 2: 1000 registros inseridos/atualizados.
   - Lote 3: 1000 registros inseridos/atualizados.
   - Lote 4: 1000 registros inseridos/atualizados.
   - Lote 5: 1000 registros inseridos/atualizados.
   - Lote 6: 392 registros inseridos/atualizados.
✅ Processamento da consistência anual concluído. Total upserted: 5392
